In [5]:
# Flight Delay Model Notebook
# This notebook rebuilds the full pipeline in one place:
# 1) Load dataset
# 2) Clean and prepare features
# 3) Train CatBoost model
# 4) Evaluate and save artifacts

In [10]:
%pip install -q catboost nycflights13 setuptools

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score

from catboost import CatBoostRegressor

# Prefer workspace root; fallback to current directory.
workspace_candidate = Path(r"d:/Personal/ADA-GWU Master/BigData&CloudComputing/Final Project")
PROJECT_ROOT = workspace_candidate if workspace_candidate.exists() else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "raw"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

for p in [DATA_DIR, MODELS_DIR, ARTIFACTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: d:\Personal\ADA-GWU Master\BigData&CloudComputing\Final Project


In [15]:
# Try local dataset first; if missing, load directly from installed nycflights13 data files.
import sys

flights_path = DATA_DIR / "nyc_flights.csv"
weather_path = DATA_DIR / "nyc_weather.csv"
package_data_dir = Path(sys.prefix) / "Lib" / "site-packages" / "nycflights13" / "data"
flights_package_path = package_data_dir / "flights.csv.zip"
weather_package_path = package_data_dir / "weather.csv"

if flights_path.exists() and weather_path.exists():
    flights = pd.read_csv(flights_path)
    weather = pd.read_csv(weather_path)
    source = "local csv"
else:
    flights = pd.read_csv(flights_package_path)
    weather = pd.read_csv(weather_package_path)
    flights.to_csv(flights_path, index=False)
    weather.to_csv(weather_path, index=False)
    source = "installed nycflights13 data files"

print("Loaded source:", source)
print("Flights shape:", flights.shape)
print("Weather shape:", weather.shape)

Loaded source: installed nycflights13 data files
Flights shape: (336776, 19)
Weather shape: (26115, 15)


In [16]:
# Data cleaning and feature preparation
required_flight_cols = [
    "year", "month", "day", "hour", "carrier", "origin", "dest", "dep_delay"
]
required_weather_cols = [
    "origin", "year", "month", "day", "hour", "wind_speed", "precip", "visib"
]

flights = flights[required_flight_cols].copy()
weather = weather[required_weather_cols].copy()

flights["dep_delay"] = pd.to_numeric(flights["dep_delay"], errors="coerce")
flights["hour"] = pd.to_numeric(flights["hour"], errors="coerce")
flights = flights.dropna(subset=["dep_delay", "hour", "origin", "dest", "carrier"])
flights["hour"] = flights["hour"].astype(int)

flights["event_time"] = pd.to_datetime(flights[["year", "month", "day"]], errors="coerce")
flights = flights.dropna(subset=["event_time"])
flights["day_of_week"] = flights["event_time"].dt.dayofweek + 1

weather["hour"] = pd.to_numeric(weather["hour"], errors="coerce")
weather = weather.dropna(subset=["hour", "origin"])
weather["hour"] = weather["hour"].astype(int)

precip = weather["precip"].fillna(0.0).clip(lower=0.0, upper=1.0)
wind = (weather["wind_speed"].fillna(0.0) / 40.0).clip(lower=0.0, upper=1.0)
visib_penalty = ((10.0 - weather["visib"].fillna(10.0)) / 10.0).clip(lower=0.0, upper=1.0)
weather["weather_severity"] = 0.5 * precip + 0.3 * wind + 0.2 * visib_penalty

flights["airport_congestion"] = flights.groupby(["origin", "year", "month", "day", "hour"])["dep_delay"].transform("count")

merged = flights.merge(
    weather[["origin", "year", "month", "day", "hour", "weather_severity"]],
    on=["origin", "year", "month", "day", "hour"],
    how="left",
)
merged["weather_severity"] = merged["weather_severity"].fillna(0.0)

prepared = pd.DataFrame(
    {
        "weather_severity": merged["weather_severity"].astype(float),
        "airport_congestion": merged["airport_congestion"].astype(float),
        "day_of_week": merged["day_of_week"].astype(int),
        "month": merged["month"].astype(int),
        "airline_code": merged["carrier"].astype(str),
        "origin": merged["origin"].astype(str),
        "destination": merged["dest"].astype(str),
        "delay_minutes": np.maximum(merged["dep_delay"].astype(float), 0.0),
    }
).replace([np.inf, -np.inf], np.nan).dropna()

print("Prepared shape:", prepared.shape)
prepared.head()

Prepared shape: (328521, 8)


,weather_severity,airport_congestion,day_of_week,month,airline_code,origin,destination,delay_minutes
0,0.094939,2.0,2,1,UA,EWR,IAH,2.0
1,0.112201,1.0,2,1,UA,LGA,IAH,4.0
2,0.112201,3.0,2,1,AA,JFK,MIA,2.0
3,0.112201,3.0,2,1,B6,JFK,BQN,0.0
4,0.120832,17.0,2,1,DL,LGA,ATL,0.0


In [20]:
# Optional sampling for faster experimentation
MAX_ROWS = 150_000
if len(prepared) > MAX_ROWS:
    prepared_model = prepared.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)
else:
    prepared_model = prepared.copy()

print("Training rows:", len(prepared_model))

Training rows: 150000


## Training Steps

1. Select the feature columns and target variable.
2. Split the data into training and validation sets.
3. Train the CatBoost regression model.
4. Evaluate the model on validation data.
5. Save the trained model and metrics to disk.

In [21]:
# Train CatBoost regressor
prepared_model = globals().get("prepared_model", prepared)

print("Step 1/5: defining features and target")
feature_cols = [
    "weather_severity",
    "airport_congestion",
    "day_of_week",
    "month",
    "airline_code",
    "origin",
    "destination",
]
target_col = "delay_minutes"
cat_cols = ["airline_code", "origin", "destination"]

print("Step 2/5: building X and y")
X = prepared_model[feature_cols].copy()
y = prepared_model[target_col].astype(float)

print("Step 3/5: splitting train and validation sets")
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cat_idx = [X.columns.get_loc(c) for c in cat_cols]
print(f"Training rows: {len(X_train)}")
print(f"Validation rows: {len(X_valid)}")
print(f"Categorical columns: {cat_cols}")

print("Step 4/5: training CatBoost model")
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=600,
    depth=8,
    learning_rate=0.05,
    random_seed=42,
    verbose=False,
)

model.fit(X_train, y_train, cat_features=cat_idx)
print("Model trained.")

print("Step 5/5: model is ready for evaluation and saving")

Step 1/5: defining features and target
Step 2/5: building X and y
Step 3/5: splitting train and validation sets
Training rows: 120000
Validation rows: 30000
Categorical columns: ['airline_code', 'origin', 'destination']
Step 4/5: training CatBoost model
Model trained.
Step 5/5: model is ready for evaluation and saving


In [22]:
# Evaluate and persist artifacts
pred_valid = model.predict(X_valid)
mae = float(mean_absolute_error(y_valid, pred_valid))
rmse = float(np.sqrt(mean_squared_error(y_valid, pred_valid)))
medae = float(median_absolute_error(y_valid, pred_valid))
r2 = float(r2_score(y_valid, pred_valid))

safe_denominator = np.maximum(np.abs(y_valid.to_numpy()), 1.0)
mape = float(np.mean(np.abs((y_valid.to_numpy() - pred_valid) / safe_denominator)) * 100.0)

metrics = {
    "sample_count": float(len(prepared_model)),
    "train_rows": float(len(X_train)),
    "valid_rows": float(len(X_valid)),
    "mae": mae,
    "rmse": rmse,
    "median_absolute_error": medae,
    "r2": r2,
    "mape_percent": mape,
}

model_path = MODELS_DIR / "catboost_delay_model.cbm"
metrics_path = ARTIFACTS_DIR / "catboost_metrics.json"

model.save_model(model_path)
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print("Evaluation Summary")
print("-" * 40)
print(f"Rows (train/valid): {int(metrics['train_rows'])} / {int(metrics['valid_rows'])}")
print(f"MAE:  {metrics['mae']:.4f}")
print(f"RMSE: {metrics['rmse']:.4f}")
print(f"MedAE:{metrics['median_absolute_error']:.4f}")
print(f"R2:   {metrics['r2']:.4f}")
print(f"MAPE: {metrics['mape_percent']:.2f}%")
print("-" * 40)
print("Saved model:", model_path)
print("Saved metrics:", metrics_path)

metrics

Evaluation Summary
----------------------------------------
Rows (train/valid): 120000 / 30000
MAE:  19.8716
RMSE: 35.6736
MedAE:12.0753
R2:   0.1044
MAPE: 878.02%
----------------------------------------
Saved model: d:\Personal\ADA-GWU Master\BigData&CloudComputing\Final Project\models\catboost_delay_model.cbm
Saved metrics: d:\Personal\ADA-GWU Master\BigData&CloudComputing\Final Project\artifacts\catboost_metrics.json


{'sample_count': 150000.0,
 'train_rows': 120000.0,
 'valid_rows': 30000.0,
 'mae': 19.871596563169902,
 'rmse': 35.67363235116869,
 'median_absolute_error': 12.075274643221459,
 'r2': 0.10436580347021418,
 'mape_percent': 878.0236529453052}

In [23]:
# Prediction code: load saved model if needed, then predict new records
from catboost import CatBoostRegressor

saved_model_path = MODELS_DIR / "catboost_delay_model.cbm"

predictor = model if "model" in globals() else CatBoostRegressor()
if "model" not in globals():
    predictor.load_model(saved_model_path)

new_flights = pd.DataFrame(
    [
        {
            "weather_severity": 0.20,
            "airport_congestion": 8.0,
            "day_of_week": 3,
            "month": 4,
            "airline_code": "AA",
            "origin": "JFK",
            "destination": "MIA",
        },
        {
            "weather_severity": 0.65,
            "airport_congestion": 17.0,
            "day_of_week": 5,
            "month": 12,
            "airline_code": "DL",
            "origin": "LGA",
            "destination": "ATL",
        },
        {
            "weather_severity": 0.05,
            "airport_congestion": 4.0,
            "day_of_week": 2,
            "month": 8,
            "airline_code": "UA",
            "origin": "EWR",
            "destination": "ORD",
        },
    ]
)

new_flights["predicted_delay_minutes"] = np.maximum(
    predictor.predict(new_flights[feature_cols]),
    0.0,
)

print("Prediction Results")
print("-" * 60)
print(new_flights[["airline_code", "origin", "destination", "predicted_delay_minutes"]])

new_flights

Prediction Results
------------------------------------------------------------
  airline_code origin destination  predicted_delay_minutes
0           AA    JFK         MIA                39.129956
1           DL    LGA         ATL                16.808053
2           UA    EWR         ORD                10.570876


,weather_severity,airport_congestion,day_of_week,month,airline_code,origin,destination,predicted_delay_minutes
0,0.20,8.0,3,4,AA,JFK,MIA,39.129956
1,0.65,17.0,5,12,DL,LGA,ATL,16.808053
2,0.05,4.0,2,8,UA,EWR,ORD,10.570876
